# Claude Agent SDK: skills and commands

A **skill** is a folder with a `SKILL.md` file: instructions Claude can load **when relevant**. A **command** is anything you run by sending `/<name>` in a prompt. In the SDK these two ideas overlap: every user-invocable skill is also a command.

**What you will learn (and see running):**
1. How skills are **discovered** (files on disk + `setting_sources`) and **loaded lazily**
2. Writing a skill and **confirming it loaded** (`init` message: `skills`, `slash_commands`)
3. **Model-invoked** skills (Claude decides) vs **user-invoked** (`/name`)
4. The **`skills` option**: `"all"`, a list, `[]`, and what an unlisted skill does
5. **Dispatching commands** by name, unknown commands, `/compact`, `/clear`
6. **Custom command files** (`.claude/commands/`), the older form
7. **Pre-approving tools** for skills
8. Troubleshooting, including the **invalid skill name** error

## The picture

```
 .claude/skills/security-check/SKILL.md      ─┐  discovered at startup (only name + description are read)
 .claude/skills/glossary/SKILL.md            ─┤  when setting_sources includes "project" / "user"
 .claude/commands/hello.md  (older style)    ─┘
                │
                ▼
   Claude sees:  [security-check: "Run a security scan", glossary: "Company terms", ...]
        ├─ model-invoked:  request matches a description ──▶ Skill tool ──▶ full SKILL.md loads
        └─ user-invoked :  prompt "/security-check"      ──▶ SKILL.md becomes the prompt
```

| Concept | Key fact |
|---|---|
| Skills live on **disk**, not in code | The SDK has **no API to register a skill**. (Subagents, by contrast, can be defined in code.) |
| Loaded **lazily** | Only metadata at startup; the full text loads when the skill is invoked |
| Location | `<project>/.claude/skills/<name>/SKILL.md` (project) or `~/.claude/skills/<name>/SKILL.md` (personal) |
| Needs a setting source | `project` / `user` must be in `setting_sources` (the default includes both) |
| The `description` decides | Claude uses it to pick the skill: make it specific |

## 0. Setup

Run these four cells first. They check Python, install the SDK, load your API key from a `.env` file, and create a private working folder for the demos.

In [ ]:
import sys
print(sys.version)
assert sys.version_info >= (3, 10), "The Agent SDK needs Python 3.10 or newer"

In [ ]:
%pip install -q claude-agent-sdk python-dotenv 

Create a file named `.env` next to this notebook containing one line (no quotes, no spaces around `=`):

```
ANTHROPIC_API_KEY=your-api-key
```

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv, find_dotenv

env_path = find_dotenv(usecwd=True)
load_dotenv(env_path, override=True)

key = os.environ.get("ANTHROPIC_API_KEY", "")
if not key:
    raise RuntimeError(
        f"ANTHROPIC_API_KEY not found. Create a .env file in {Path.cwd()} with:\n"
        "ANTHROPIC_API_KEY=your-api-key"
    )
print("Loaded .env from:", env_path)
print("API key loaded:", key[:7] + "..." + key[-4:])

In [ ]:
from pathlib import Path

DEMO_DIR = Path("skills-demo").resolve()
DEMO_DIR.mkdir(exist_ok=True)
print("Demo folder:", DEMO_DIR)

## 1. Imports and a runner

`run()` prints the key facts from the **`init` message** (loaded skills and slash commands), every **Skill tool** call, and the answer.

In [ ]:
import asyncio, json, textwrap
from pathlib import Path

from claude_agent_sdk import (
    query, ClaudeAgentOptions, ClaudeSDKClient,
    AssistantMessage, UserMessage, ResultMessage, SystemMessage,
    ToolUseBlock, ToolResultBlock,
)

BUNDLED = {"clear", "compact", "context", "usage", "code-review", "verify", "init", "review"}

async def run(prompt, show_init=False, **opts):
    opts.setdefault("cwd", str(DEMO_DIR))
    opts.setdefault("max_turns", 8)
    opts.setdefault("setting_sources", ["project"])
    final, calls, init = "", [], {}
    try:
        async for m in query(prompt=prompt, options=ClaudeAgentOptions(**opts)):
            if isinstance(m, SystemMessage) and m.subtype == "init":
                init = m.data
                if show_init:
                    print("  skills        :", init.get("skills"))
                    print("  slash_commands:", init.get("slash_commands"))
            elif isinstance(m, AssistantMessage):
                for b in m.content:
                    if isinstance(b, ToolUseBlock):
                        calls.append(b.name)
                        arg = b.input.get("skill") or b.input.get("command") or b.input.get("file_path") or b.input.get("pattern") or ""
                        print(f"  [tool] {b.name} {str(arg)[:70]}")
            elif isinstance(m, UserMessage) and isinstance(m.content, list):
                for b in m.content:
                    if isinstance(b, ToolResultBlock) and b.is_error:
                        print("  [tool error]", str(b.content)[:150])
            elif isinstance(m, ResultMessage):
                final = m.result if m.subtype == "success" else f"({m.subtype})"
    except Exception as error:
        final = f"(raised {type(error).__name__}: {str(error)[:200]})"
    print(textwrap.indent((final or "").strip()[:700], "  | "))
    return final, calls, init

## 2. Create a small project with skills

We write four things into the demo folder:

| File | Purpose |
|---|---|
| `app.py` | a tiny app with **seeded security flaws** to scan |
| `.claude/skills/security-check/SKILL.md` | an **action** skill you can dispatch with `/security-check` |
| `.claude/skills/acme-glossary/SKILL.md` | a **knowledge** skill Claude should pick up on its own |
| `.claude/commands/hello.md` | an older-style **command file** |

A `SKILL.md` is YAML frontmatter (`name`, `description`) followed by Markdown instructions. The `description` is what Claude reads to decide whether to use it, so say *what it does and when*.

In [ ]:
(DEMO_DIR / "app.py").write_text("""import sqlite3
PASSWORD = "hunter2-prod-db-password"   # hard-coded credential

def find_user(db, name):
    cur = db.cursor()
    cur.execute("SELECT * FROM users WHERE name = '" + name + "'")   # SQL built by concatenation
    return cur.fetchall()
""")

skills = DEMO_DIR / ".claude" / "skills"
(skills / "security-check").mkdir(parents=True, exist_ok=True)
(skills / "security-check" / "SKILL.md").write_text("""---
name: security-check
description: Run a security vulnerability scan on the codebase (SQL injection, exposed credentials, insecure configuration)
---

Analyze the code in the current directory for security vulnerabilities including:
- SQL injection risks
- Exposed credentials
- Insecure configurations

Report each finding as: `file:line - issue - one-line fix`. Start your reply with the exact text `SECURITY REPORT`.
""")

(skills / "acme-glossary").mkdir(parents=True, exist_ok=True)
(skills / "acme-glossary" / "SKILL.md").write_text("""---
name: acme-glossary
description: Definitions of internal Acme company terms and jargon (for example 'blorp' and 'snazzle'). Use when the user asks what an Acme term means.
---

Acme internal glossary:
- **blorp**: a deployment that was rolled back within 10 minutes of release.
- **snazzle**: the weekly Friday meeting where all blorps are reviewed.

When you use this glossary, begin your answer with the exact text `[ACME-GLOSSARY]`.
""")

(skills / "internal-helper").mkdir(parents=True, exist_ok=True)
(skills / "internal-helper" / "SKILL.md").write_text("""---
name: internal-helper
description: Internal helper that Claude may use, but that users cannot dispatch by name.
user-invocable: false
---

Helper instructions (not meant to be run by a user).
""")

commands = DEMO_DIR / ".claude" / "commands"
commands.mkdir(parents=True, exist_ok=True)
(commands / "hello.md").write_text("""Greet the user warmly in exactly one short sentence, and mention the word "commands".
""")

for p in sorted(DEMO_DIR.rglob("*.md")):
    print(p.relative_to(DEMO_DIR))

## 3. Confirm what loaded (the `init` message)

The first message of every query is a `system` message with subtype `init`. Two fields matter:

| Field | Contains |
|---|---|
| `skills` | user-invocable skills with a `description`, plus **bundled** skills shipped with Claude Code |
| `slash_commands` | everything you can dispatch with `/name`: built-in commands, bundled skills, **your skills**, and `.claude/commands/` files |

A skill marked `user-invocable: false` still **loads and is available to Claude**, but is **hidden from both lists**. We should see that: `internal-helper` is absent.

In [ ]:
_, _, init = await run("Reply with the single word: ready", show_init=True, max_turns=1)

skills_seen = init.get("skills", [])
cmds_seen = init.get("slash_commands", [])
print()
for name in ["security-check", "acme-glossary", "internal-helper"]:
    print(f"  {name:16} in skills: {name in skills_seen!s:5}  in slash_commands: {name in cmds_seen}")
print("  hello (command file) in slash_commands:", "hello" in cmds_seen)

## 4. Model-invoked skills: Claude chooses

We never say "use the glossary skill". The question just *matches the description*. Look for a **`Skill`** tool call and the `[ACME-GLOSSARY]` marker the skill told Claude to print. Claude could not know what a "blorp" is otherwise.

In [ ]:
answer, calls, _ = await run("At Acme, what does the word 'blorp' mean?")
print("\n  used Skill tool:", "Skill" in calls, "| marker present:", "[ACME-GLOSSARY]" in answer)

## 5. User-invoked skills: dispatch with `/name`

Send `/<name>` as the prompt. The skill's instructions become the task. Here `/security-check` scans `app.py` for the planted flaws. Because the skill reads files, we pre-approve the read-only tools (section 9 explains this).

In [ ]:
answer, calls, _ = await run("/security-check", allowed_tools=["Read", "Grep", "Glob"], max_turns=10)
print("\n  starts with SECURITY REPORT:", answer.strip().startswith("SECURITY REPORT"))
print("  mentions SQL injection:", "sql" in answer.lower(), "| mentions the password:", "password" in answer.lower())

## 6. The `skills` option: who may invoke what

| `skills=` | Effect |
|---|---|
| *(omitted)* | all discovered skills enabled, `Skill` tool available (like the CLI) |
| `"all"` | every discovered skill (use this, **not** a wildcard) |
| `["a", "b"]` | only these; others are **hidden from the model** and the Skill tool rejects them |
| `[]` | Claude can invoke none |

Setting `skills` also **adds the `Skill` tool to `allowed_tools`** for you. (If you pass an explicit `tools=` list, include `"Skill"` in it.)

Important caveats:
- Unlisted skills' **files stay on disk**: Claude can still `Read` them with file tools.
- Restricting `skills` does **not** restrict `/name` dispatch (section 7).
- **It is a visibility filter, not a security boundary.** In an earlier run of this very experiment, with file tools blocked but `Agent` allowed, Claude went around the filter through a subagent and still found the unlisted skill. Treat `skills` as steering; use `tools` / `disallowed_tools` / permissions for real restrictions.

Same glossary question, three configurations. We block file tools **and the `Agent` tool** so the *Skill tool* is the only way in, and we test whether the answer is **known** (only the glossary mentions that a snazzle is a Friday meeting):

In [ ]:
Q = "At Acme, what does the word 'snazzle' mean?"
block_files = dict(disallowed_tools=["Read", "Glob", "Grep", "Bash", "Agent"])   # Agent blocked too: a subagent is another way in

print("skills=['acme-glossary']")
a1, c1, _ = await run(Q, skills=["acme-glossary"], **block_files)
print("\nskills=['security-check']  (glossary NOT listed)")
a2, c2, _ = await run(Q, skills=["security-check"], **block_files)
print("\nskills=[]  (none)")
a3, c3, _ = await run(Q, skills=[], **block_files)

def knows(text): return "friday" in text.lower()      # only the glossary skill knows what a snazzle is
print()
print("knows the answer when listed         :", knows(a1))
print("knows the answer when NOT listed     :", knows(a2))
print("knows the answer with skills=[]      :", knows(a3))

## 7. Dispatching commands by name

Dispatch does **not** depend on the `skills` option: `/<name>` runs any user-invocable skill even if your `skills` list omits it.

What happens with names that do not match:

| You send | Result |
|---|---|
| `/security-check` (a skill) | runs it, even if not in `skills` |
| `/hello` (command file) | runs the file's prompt |
| `/does-not-exist` | **does not fail**: sent to Claude as an ordinary message with a note that the command did not run (costs a model turn) |
| `/theme` (a built-in that needs a terminal) | returns `/theme isn't available in this environment.` with **no** model turn |

Also, a command can hit `max_turns` like any prompt and end with an **error result** instead of `success`; set `max_turns` high enough or wrap the loop in `try/except`.

In [ ]:
print("/hello  (command file), skills=[] so the Skill tool is not even enabled:")
await run("/hello", skills=[])

print("\n/does-not-exist :")
await run("/does-not-exist", max_turns=2)

print("\n/theme :")
await run("/theme", max_turns=1)

## 8. `/compact` and `/clear`

Both act on **conversation history**, so they need a session with prior turns.

- **`/compact`** summarizes older messages to save context. When it runs you get a `compact_boundary` system message with `compact_metadata` (`pre_tokens`, `trigger`). If there is nothing to summarize, you get a normal `success` result such as *"Not enough messages to compact."* and **no** boundary message.
- **`/clear`** resets to an empty context (the old conversation stays on disk and can be resumed by id). It is useful in **streaming input** sessions; for a one-shot `query()` it has no practical effect because every call already starts empty.

We build a short conversation, then compact it:

In [ ]:
async def compact_demo():
    # a few turns of history
    for text in ["Remember this number: 4821. Reply 'noted'.", "Also remember the word 'plum'. Reply 'noted'."]:
        async for m in query(prompt=text, options=ClaudeAgentOptions(cwd=str(DEMO_DIR), setting_sources=[], tools=[],
                                                                    continue_conversation=True, max_turns=2)):
            pass
    boundary = None
    result = ""
    async for m in query(prompt="/compact", options=ClaudeAgentOptions(cwd=str(DEMO_DIR), setting_sources=[], tools=[],
                                                                      continue_conversation=True, max_turns=2)):
        if isinstance(m, SystemMessage) and m.subtype == "compact_boundary":
            boundary = m.data.get("compact_metadata")
        elif isinstance(m, ResultMessage):
            result = m.result if m.subtype == "success" else m.subtype
    print("compact_boundary received:", boundary)
    print("result text:", (result or "")[:200])

await compact_demo()

(If you see no boundary, read the result text: Claude Code tells you why nothing was compacted. The mechanics that matter: check for the `compact_boundary` message instead of assuming compaction happened.)

## 9. Pre-approving tools for skills

Skills run with the **session's tools**. In a script nobody can click "Approve", so pre-approve what the skill needs. Two equivalent routes:

1. the skill's own frontmatter: `allowed-tools: Read Grep Glob`
2. `allowed_tools=["Read", "Grep", "Glob"]` in your options (what we did in section 5)

`allowed_tools` **pre-approves**; it does not **restrict** the other tools (use `tools=` / `disallowed_tools` for that). If your organization sets `allowManagedPermissionRulesOnly` in managed settings, both are ignored.

## 10. Troubleshooting

| Symptom | Check |
|---|---|
| Skill not found | `setting_sources` must include `"project"` / `"user"`; `[]` loads **no** skills |
| Still not found | `cwd` must be at or below the folder containing `.claude/skills/` (parents up to the repo root are searched) |
| Not in `init.skills` | Is it `user-invocable: false`? Is the `description` frontmatter present? |
| Claude never uses it | Vague `description`. Add the nouns/phrases users actually say. Also check the `skills` list includes it |
| `Skill <name> is not in this session's skills allowlist` | You listed other skills but not this one: add it, or dispatch `/name` |
| `ls .claude/skills/*/SKILL.md` | quick filesystem sanity check |

Prove that `setting_sources=[]` hides skills:

In [ ]:
_, _, init_none = await run("Reply with the single word: ready", max_turns=1, setting_sources=[])
print("with setting_sources=[]  -> project skills visible:", "security-check" in init_none.get("skills", []))

### The invalid skill name error

Names in `skills=[...]` must be **exact**. These are rejected **before** the Claude Code process even starts:
- an empty name
- names with parentheses, commas, or control characters
- names padded with whitespace
- wildcards: a bare `*` or a `:*` suffix (use `skills="all"` for everything)

Python raises `ValueError`:

In [ ]:
# OFFLINE
async def bad(names):
    try:
        async for _ in query(prompt="hi", options=ClaudeAgentOptions(skills=names, cwd=str(DEMO_DIR))):
            pass
        return "no error"
    except Exception as e:
        return f"{type(e).__name__}: {str(e)[:110]}"

for names in (["docs:*"], [""], [" padded "], ["a,b"]):
    print(f"skills={names!r:16} -> {await bad(names)}")

## 11. Skills vs commands vs subagents vs CLAUDE.md

| | Defined in | Triggered by | Loads | Good for |
|---|---|---|---|---|
| **Skill** | `.claude/skills/<n>/SKILL.md` | Claude (description match) **or** you (`/n`) | lazily, on use | reusable procedures and knowledge |
| **Command file** | `.claude/commands/n.md` | you (`/n`) | on dispatch | older style; same behaviour as a skill |
| **Built-in command** | Claude Code itself | you (`/compact`, `/clear`) | n/a | session housekeeping |
| **Subagent** | code (`agents=`) or files | Claude delegates | own context | isolating a subtask |
| **CLAUDE.md** | project root | always | every session | standing project rules |

## 12. Summary

| I want to ... | Do this |
|---|---|
| Add a reusable capability | `.claude/skills/<name>/SKILL.md` with a specific `description` |
| Let Claude use it automatically | nothing else (default) |
| Run it myself | prompt `"/<name>"` |
| Limit which skills Claude may invoke | `skills=["a", "b"]` or `[]` (remember: files stay readable, `/name` still works) |
| Hide a helper from users | `user-invocable: false` |
| Check it loaded | `init.skills` and `init.slash_commands` |
| Let a skill read files unattended | `allowed_tools=["Read", "Grep", "Glob"]` |
| Shrink a long session | `"/compact"` + look for `compact_boundary` |

**Related:** `agent-sdk-system-prompts-explained` (CLAUDE.md, output styles, append) and `agent-sdk-claude-code-features-explained`.